In [1]:
import os, sys
import numpy as np

from sklearn.metrics import roc_auc_score
from scipy.stats import t

import project_dirs as pdir

In [2]:
results_root_dir = pdir.RESULT_DIR

## "Slide_MetaData_resul

Folds = [0, 1, 2]

In [3]:
index_to_label = {
    0: 'Acinar',
    1: 'Cribriform',
    2: 'In situ',
    3: 'Lepidic',
    4: 'Micropapillary',
    5: 'Papillary',
    6: 'Solid'
}

label_to_index = {
    label: index for index, label in index_to_label.items()
}

In [4]:
def summarize_folds(results, confidence=0.95):
    results = np.asarray(results, dtype=float)

    n = len(results)
    mean = np.mean(results)
    std = np.std(results, ddof=1)  # sample standard deviation

    # t-based confidence interval
    alpha = 1 - confidence
    t_critical = t.ppf(1 - alpha / 2, df=n - 1)

    standard_error = std / np.sqrt(n)
    margin = t_critical * standard_error

    ci_lower = mean - margin
    ci_upper = mean + margin

    return {
        "mean": mean,
        "std": std,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "ci": (ci_lower, ci_upper)
    }

In [5]:
def compute_auc_roc(y_true, y_pred, y_logits, label_to_index):
    """
    Compute per-class ROC-AUC and macro-average ROC-AUC.

    Parameters
    ----------
    y_true : list
        Ground-truth class labels, e.g. ['In situ', 'In situ', 'Acinar', ...]

    y_pred : list
        Predicted class labels, e.g. ['In situ', 'In situ', 'In situ', ...]

    y_logits : np.ndarray or torch.Tensor
        Model output logits with shape (N, num_classes).

    label_to_index : dict
        Mapping from class label to integer index.

    Returns
    -------
    per_class_auc : dict
        ROC-AUC for each class.

    macro_auc : float
        Macro-average ROC-AUC across classes.
    """

    # Convert logits to numpy
    if hasattr(y_logits, "detach"):
        y_logits = y_logits.detach().cpu().numpy()
    else:
        y_logits = np.asarray(y_logits)

    # Convert labels to indices
    y_true_idx = np.array([
        label_to_index[label] for label in y_true
    ])

    y_pred_idx = np.array([
        label_to_index[label] for label in y_pred
    ])

    # Convert logits to probabilities using softmax
    # ROC-AUC can technically use logits directly, but probabilities
    # are more intuitive and standard for multiclass classification.
    exp_logits = np.exp(
        y_logits - np.max(y_logits, axis=1, keepdims=True)
    )
    y_prob = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    # Number of classes
    num_classes = len(label_to_index)

    # One-hot encode ground truth
    y_true_onehot = np.eye(num_classes)[y_true_idx]

    # Compute AUC for each class
    per_class_auc = {}

    for label, class_idx in label_to_index.items():

        # Check whether this class is present in y_true
        if len(np.unique(y_true_onehot[:, class_idx])) < 2:
            per_class_auc[label] = np.nan
            continue

        auc = roc_auc_score(
            y_true_onehot[:, class_idx],
            y_prob[:, class_idx]
        )

        per_class_auc[label] = auc

    # Macro AUC
    valid_auc = [
        auc for auc in per_class_auc.values()
        if not np.isnan(auc)
    ]

    macro_auc = np.mean(valid_auc)

    return per_class_auc, macro_auc

# MetaData Only (Sex + Age)

In [6]:

macro_f1_scores = []
bal_accu_scores = []
macro_auc_scores = []

metadata_model = "AgeSex_Linear"

for fold in Folds:

    results_dir = os.path.join(results_root_dir, "Metadata_Classifier_results", metadata_model, f"Fold_{fold}") ### for Clinical data

    result_eval = np.load(os.path.join(results_dir, f"{metadata_model}_eval.npy"), allow_pickle=True)
    result_eval = result_eval.item()

    result_dict = np.load(os.path.join(results_dir, f"{metadata_model}_results.npy"), allow_pickle=True)
    result_dict = result_dict.item()

    bal_accu_scores.append(result_dict['balanced_accuracy'])
    macro_f1_scores.append(result_dict['f1_macro']*100)

    y_true = result_eval["True_Labels"]
    y_pred = result_eval["Pred_Labels"]
    y_logits = result_eval["Logits"]

    per_class_auc, macro_auc = compute_auc_roc(y_true, y_pred, y_logits, label_to_index)
    macro_auc_scores.append(macro_auc*100)




## Macro AU-ROC

In [7]:
summary_auc = summarize_folds(macro_auc_scores)
print(f"Mean: {summary_auc['mean']:.4f}")
print(f"Std:  {summary_auc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_auc['ci_lower']:.4f}, {summary_auc['ci_upper']:.4f}]")

Mean: 61.0806
Std:  3.0491
95% CI: [53.5062, 68.6549]


## Balanced Accuracy

In [8]:
summary_bal_acc = summarize_folds(bal_accu_scores)
print(f"Mean: {summary_bal_acc['mean']:.4f}")
print(f"Std:  {summary_bal_acc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_bal_acc['ci_lower']:.4f}, {summary_bal_acc['ci_upper']:.4f}]")

Mean: 23.9533
Std:  2.1015
95% CI: [18.7330, 29.1737]


## Macro F1 Scores

In [9]:
summary_f1_score = summarize_folds(macro_f1_scores)
print(f"Mean: {summary_f1_score['mean']:.4f}")
print(f"Std:  {summary_f1_score['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_f1_score['ci_lower']:.4f}, {summary_f1_score['ci_upper']:.4f}]")

Mean: 17.3122
Std:  1.4337
95% CI: [13.7506, 20.8739]


# MIL Models

In [10]:

macro_f1_scores = []
bal_accu_scores = []
macro_auc_scores = []

model_name = "H-OPTIMUS-1"

print(model_name)

for fold in Folds:

    results_dir = os.path.join(results_root_dir, "MIL_results", f"{model_name}_ABMIL_8", model_name, f"Fold_{fold}")

    result_eval = np.load(os.path.join(results_dir, f"{model_name}_eval.npy"), allow_pickle=True)
    result_eval = result_eval.item()

    result_dict = np.load(os.path.join(results_dir, f"{model_name}_results.npy"), allow_pickle=True)
    result_dict = result_dict.item()

    bal_accu_scores.append(result_dict['balanced_accuracy'])
    macro_f1_scores.append(result_dict['f1_macro']*100)

    y_true = result_eval["True_Labels"]
    y_pred = result_eval["Pred_Labels"]
    y_logits = result_eval["Logits"]

    per_class_auc, macro_auc = compute_auc_roc(y_true, y_pred, y_logits, label_to_index)
    macro_auc_scores.append(macro_auc*100)


H-OPTIMUS-1


## Macro AU-ROC

In [11]:
print(model_name)
summary_auc = summarize_folds(macro_auc_scores)
print(f"Mean: {summary_auc['mean']:.4f}")
print(f"Std:  {summary_auc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_auc['ci_lower']:.4f}, {summary_auc['ci_upper']:.4f}]")

H-OPTIMUS-1
Mean: 88.2448
Std:  2.1941
95% CI: [82.7943, 93.6952]


## Balanced Accuracy

In [12]:
print(model_name)
summary_bal_acc = summarize_folds(bal_accu_scores)
print(f"Mean: {summary_bal_acc['mean']:.4f}")
print(f"Std:  {summary_bal_acc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_bal_acc['ci_lower']:.4f}, {summary_bal_acc['ci_upper']:.4f}]")

H-OPTIMUS-1
Mean: 61.8133
Std:  4.9155
95% CI: [49.6025, 74.0242]


## Macro F1 Scores

In [13]:
print(model_name)
summary_f1_score = summarize_folds(macro_f1_scores)
print(f"Mean: {summary_f1_score['mean']:.4f}")
print(f"Std:  {summary_f1_score['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_f1_score['ci_lower']:.4f}, {summary_f1_score['ci_upper']:.4f}]")

H-OPTIMUS-1
Mean: 61.9592
Std:  4.9484
95% CI: [49.6667, 74.2518]


# MIL Models + MetaData (Sex + Age)

In [14]:

macro_f1_scores = []
bal_accu_scores = []
macro_auc_scores = []

model_name = "H-OPTIMUS-1"

print(f"{model_name} ABMIL + {metadata_model}")

for fold in Folds:

    results_dir = os.path.join(results_root_dir, "Slide_MetaData_results", f"{model_name}_ABMIL_{metadata_model}", f"ABMIL_{metadata_model}", f"Fold_{fold}")

    result_eval = np.load(os.path.join(results_dir, f"ABMIL_{metadata_model}_eval.npy"), allow_pickle=True)
    result_eval = result_eval.item()

    result_dict = np.load(os.path.join(results_dir, f"ABMIL_{metadata_model}_results.npy"), allow_pickle=True)
    result_dict = result_dict.item()

    bal_accu_scores.append(result_dict['balanced_accuracy'])
    macro_f1_scores.append(result_dict['f1_macro']*100)

    y_true = result_eval["True_Labels"]
    y_pred = result_eval["Pred_Labels"]
    y_logits = result_eval["Logits"]

    per_class_auc, macro_auc = compute_auc_roc(y_true, y_pred, y_logits, label_to_index)
    macro_auc_scores.append(macro_auc*100)


H-OPTIMUS-1 ABMIL + AgeSex_Linear


## Macro AU-ROC

In [15]:
print(model_name)
summary_auc = summarize_folds(macro_auc_scores)
print(f"Mean: {summary_auc['mean']:.4f}")
print(f"Std:  {summary_auc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_auc['ci_lower']:.4f}, {summary_auc['ci_upper']:.4f}]")

H-OPTIMUS-1
Mean: 88.0672
Std:  3.4547
95% CI: [79.4851, 96.6492]


## Balanced Accuracy

In [16]:
print(model_name)
summary_bal_acc = summarize_folds(bal_accu_scores)
print(f"Mean: {summary_bal_acc['mean']:.4f}")
print(f"Std:  {summary_bal_acc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_bal_acc['ci_lower']:.4f}, {summary_bal_acc['ci_upper']:.4f}]")

H-OPTIMUS-1
Mean: 62.1300
Std:  1.9055
95% CI: [57.3965, 66.8635]


## Macro F1 Scores

In [17]:
print(model_name)
summary_f1_score = summarize_folds(macro_f1_scores)
print(f"Mean: {summary_f1_score['mean']:.4f}")
print(f"Std:  {summary_f1_score['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_f1_score['ci_lower']:.4f}, {summary_f1_score['ci_upper']:.4f}]")

H-OPTIMUS-1
Mean: 58.6888
Std:  1.2471
95% CI: [55.5908, 61.7868]


# AI Agent Model

In [18]:
def compute_auc_roc_agent(y_true, y_pred, y_logits, label_to_index):
    """
    Compute per-class ROC-AUC and macro-average ROC-AUC.

    Parameters
    ----------
    y_true : list
        Ground-truth class labels, e.g. ['In situ', 'In situ', 'Acinar', ...]

    y_pred : list
        Predicted class labels, e.g. ['In situ', 'In situ', 'In situ', ...]

    y_logits : np.ndarray or torch.Tensor
        Model output logits with shape (N, num_classes).
    Returns
    -------
    per_class_auc : dict
        ROC-AUC for each class.

    macro_auc : float
        Macro-average ROC-AUC across classes.
    """

    # Convert logits to numpy
    if hasattr(y_logits, "detach"):
        y_logits = y_logits.detach().cpu().numpy()
    else:
        y_logits = np.asarray(y_logits)

    # Convert labels to indices
    y_true_idx = y_true

    y_pred_idx = y_pred

    # Convert logits to probabilities using softmax
    # ROC-AUC can technically use logits directly, but probabilities
    # are more intuitive and standard for multiclass classification.
    exp_logits = np.exp(
        y_logits - np.max(y_logits, axis=1, keepdims=True)
    )
    y_prob = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    # Number of classes
    num_classes = len(label_to_index)

    # One-hot encode ground truth
    y_true_onehot = np.eye(num_classes)[y_true_idx]

    # Compute AUC for each class
    per_class_auc = {}

    for label, class_idx in label_to_index.items():

        # Check whether this class is present in y_true
        if len(np.unique(y_true_onehot[:, class_idx])) < 2:
            per_class_auc[label] = np.nan
            continue

        auc = roc_auc_score(
            y_true_onehot[:, class_idx],
            y_prob[:, class_idx]
        )

        per_class_auc[label] = auc

    # Macro AUC
    valid_auc = [
        auc for auc in per_class_auc.values()
        if not np.isnan(auc)
    ]

    macro_auc = np.mean(valid_auc)

    return per_class_auc, macro_auc

In [19]:
import torch 

macro_f1_scores = []
bal_accu_scores = []
macro_auc_scores = []

model_name = "AI AGENT"

print(f"{model_name}")

for fold in Folds:

    results_dir = os.path.join(results_root_dir, "Embedding_Fusion", f"Fold_{fold}")

    result_eval = np.load(os.path.join(results_dir, f"BestFusion_test_results.npy"), allow_pickle=True)
    result_eval = result_eval.item()

    result_dict = torch.load(os.path.join(results_dir, f"test_results.pt"))
    # result_dict = result_dict.item()

    bal_accu_scores.append(result_eval['balanced_accuracy'])
    macro_f1_scores.append(result_eval['f1_macro']*100)

    y_true = result_dict["True_label"]
    y_pred = result_dict["Pred_label"]
    y_logits = result_dict["logits"]

    per_class_auc, macro_auc = compute_auc_roc_agent(y_true, y_pred, y_logits, label_to_index)
    macro_auc_scores.append(macro_auc*100)


AI AGENT


## Macro AU-ROC

In [20]:
print(model_name)
summary_auc = summarize_folds(macro_auc_scores)
print(f"Mean: {summary_auc['mean']:.4f}")
print(f"Std:  {summary_auc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_auc['ci_lower']:.4f}, {summary_auc['ci_upper']:.4f}]")

AI AGENT
Mean: 88.3030
Std:  3.1714
95% CI: [80.4248, 96.1811]


## Balanced Accuracy

In [21]:
print(model_name)
summary_bal_acc = summarize_folds(bal_accu_scores)
print(f"Mean: {summary_bal_acc['mean']:.4f}")
print(f"Std:  {summary_bal_acc['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_bal_acc['ci_lower']:.4f}, {summary_bal_acc['ci_upper']:.4f}]")

AI AGENT
Mean: 63.4300
Std:  4.8152
95% CI: [51.4684, 75.3916]


## Macro F1 Scores

In [22]:
print(model_name)
summary_f1_score = summarize_folds(macro_f1_scores)
print(f"Mean: {summary_f1_score['mean']:.4f}")
print(f"Std:  {summary_f1_score['std']:.4f}")
print(
    f"95% CI: "
    f"[{summary_f1_score['ci_lower']:.4f}, {summary_f1_score['ci_upper']:.4f}]")

AI AGENT
Mean: 61.8761
Std:  7.6634
95% CI: [42.8390, 80.9131]
